# **Semana 8: Captura de Datos - Simulación de CDC (NB2 - Práctico/Ejercicios)**

## **Propósito de la Sesión**
Simular un pipeline de **Change Data Capture (CDC)** procesando un archivo de logs que registra cambios en una base de datos, y aplicando esos cambios a una base de datos SQLite local. Esta práctica representa un escenario real donde los cambios de una base de datos transaccional (como PostgreSQL o MySQL) son capturados en tiempo real y replicados en un Data Lake o Data Warehouse.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Comprender** el concepto de Change Data Capture (CDC) y su importancia en la integración de datos en tiempo real.
2. **Simular** un productor de logs de cambios (inserciones, actualizaciones, eliminaciones) en una base de datos.
3. **Procesar** un archivo de logs para detectar y clasificar cambios.
4. **Aplicar** los cambios detectados a una base de datos SQLite de destino, manteniéndola sincronizada.
5. **Reflexionar** sobre el rol del CDC en arquitecturas de datos modernas y su relación con herramientas como Debezium y Apache Kafka.

## **Configuración Inicial**

Importamos las librerías necesarias y configuramos el entorno de trabajo.

In [ ]:
# Instalación de librerías necesarias
!pip install pandas --quiet

# Importación de librerías
import pandas as pd
import sqlite3
import json
import os
import random
from datetime import datetime, timedelta
import time

print("Librerías importadas correctamente.")
print(f"Versión de pandas: {pd.__version__}")

---
## **Parte 1: Concepto de Change Data Capture (CDC)**

### **¿Qué es CDC?**

**Change Data Capture (Captura de Datos de Cambio)** es una técnica utilizada para identificar y capturar los cambios realizados en una base de datos (inserciones, actualizaciones y eliminaciones) y aplicarlos a otro sistema de destino en tiempo real o casi en tiempo real.

### **¿Por qué es importante para Big Data e IA?**

*   **Sincronización en tiempo real**: Permite mantener actualizados Data Lakes, Data Warehouses o cachés sin afectar el rendimiento de la base de datos transaccional.
*   **Alimentación de modelos de IA**: Los modelos de machine learning pueden beneficiarse de datos actualizados al minuto (ej. detección de fraude, recomendaciones personalizadas).
*   **Arquitecturas modernas**: Es el corazón de pipelines basados en **Apache Kafka** y herramientas como **Debezium**, que leen los logs transaccionales (binlog de MySQL, WAL de PostgreSQL) y publican los cambios como eventos.

### **Métodos de CDC**

1.  **Basado en logs (el más común y eficiente)**: Lee directamente los logs de transacciones de la base de datos.
2.  **Basado en triggers**: Crea triggers en la base de datos que escriben los cambios en una tabla de auditoría.
3.  **Basado en consultas periódicas**: Compara el estado actual de los datos con una versión anterior (no es tiempo real).

En este notebook, **simularemos el método basado en logs** leyendo un archivo que actúa como si fuera un log de cambios.

---
## **Parte 2: Simulación de un Productor de Logs de Cambios**

Primero, necesitamos datos de origen. Crearemos una base de datos SQLite que simule nuestra "base de datos transaccional" (origen) y generaremos un archivo de logs con los cambios que ocurren en ella.

### **2.1. Crear la Base de Datos de Origen**

Crearemos una tabla `clientes` que contendrá información básica de clientes. Esta será nuestra base de datos que sufre cambios.

In [ ]:
# Conectar a la base de datos de origen (se creará si no existe)
conn_origen = sqlite3.connect('db_origen.db')
cursor_origen = conn_origen.cursor()

# Crear tabla clientes (si no existe)
cursor_origen.execute('''
    CREATE TABLE IF NOT EXISTS clientes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        ciudad TEXT,
        fecha_registro TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        activo BOOLEAN DEFAULT 1
    )
''')

# Insertar datos iniciales (10 clientes de ejemplo)
datos_iniciales = [
    ('Ana García', 'ana.garcia@email.com', 'Madrid'),
    ('Carlos López', 'carlos.lopez@email.com', 'Barcelona'),
    ('María Rodríguez', 'maria.rodriguez@email.com', 'Valencia'),
    ('Juan Martínez', 'juan.martinez@email.com', 'Sevilla'),
    ('Laura Sánchez', 'laura.sanchez@email.com', 'Bilbao'),
    ('Pedro Gómez', 'pedro.gomez@email.com', 'Málaga'),
    ('Elena Díaz', 'elena.diaz@email.com', 'Murcia'),
    ('David Fernández', 'david.fernandez@email.com', 'Palma'),
    ('Sara Ruiz', 'sara.ruiz@email.com', 'Las Palmas'),
    ('Javier Pérez', 'javier.perez@email.com', 'Alicante')
]

for cliente in datos_iniciales:
    try:
        cursor_origen.execute(
            "INSERT INTO clientes (nombre, email, ciudad) VALUES (?, ?, ?)",
            cliente
        )
    except sqlite3.IntegrityError:
        # Si el email ya existe, ignoramos (útil para ejecuciones múltiples)
        pass

conn_origen.commit()

# Verificar los datos insertados
df_clientes_origen = pd.read_sql_query("SELECT * FROM clientes", conn_origen)
print("Datos en la base de datos ORIGEN:")
display(df_clientes_origen)

# No cerramos la conexión aún, la usaremos para simular cambios

### **2.2. Simular Cambios y Generar Archivo de Logs**

Ahora simularemos una serie de operaciones (INSERT, UPDATE, DELETE) sobre la tabla `clientes` y las registraremos en un archivo de logs. Este archivo simula el log transaccional (como el binlog de MySQL o el WAL de PostgreSQL).

In [ ]:
# Función para generar un timestamp con formato ISO
def generar_timestamp():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Nombre del archivo de logs
log_file = 'cambios_clientes.log'

# Si el archivo ya existe, lo eliminamos para empezar de cero
if os.path.exists(log_file):
    os.remove(log_file)

# Abrimos el archivo de logs en modo escritura
with open(log_file, 'w') as f:
    f.write(f"# LOG DE CAMBIOS - SIMULACIÓN CDC\n")
    f.write(f"# Generado el: {generar_timestamp()}\n")
    f.write("# FORMATO: timestamp|operacion|datos_json\n\n")

# Función para registrar un cambio en el log
def registrar_cambio(operacion, datos):
    with open(log_file, 'a') as f:
        timestamp = generar_timestamp()
        linea = f"{timestamp}|{operacion}|{json.dumps(datos, ensure_ascii=False)}\n"
        f.write(linea)
    print(f"Cambio registrado: {operacion} - {datos}")

print("--- SIMULANDO CAMBIOS EN LA BASE DE DATOS ORIGEN ---")

# --- Operación 1: INSERT (nuevo cliente) ---
nuevo_cliente = {
    'nombre': 'Roberto Santana',
    'email': 'roberto.santana@email.com',
    'ciudad': 'Granada'
}
cursor_origen.execute(
    "INSERT INTO clientes (nombre, email, ciudad) VALUES (?, ?, ?)",
    (nuevo_cliente['nombre'], nuevo_cliente['email'], nuevo_cliente['ciudad'])
)
nuevo_id = cursor_origen.lastrowid
nuevo_cliente['id'] = nuevo_id
registrar_cambio('INSERT', nuevo_cliente)
conn_origen.commit()
time.sleep(1)  # Pequeña pausa para diferenciar timestamps

# --- Operación 2: UPDATE (cambiar ciudad de un cliente existente) ---
cliente_id_update = 3  # María Rodríguez
nueva_ciudad = 'A Coruña'

# Obtener datos antes del update para el log
cursor_origen.execute("SELECT nombre, email, ciudad FROM clientes WHERE id = ?", (cliente_id_update,))
datos_antiguos = cursor_origen.fetchone()

# Realizar el update
cursor_origen.execute(
    "UPDATE clientes SET ciudad = ? WHERE id = ?",
    (nueva_ciudad, cliente_id_update)
)

# Registrar el cambio (incluyendo valores antiguos y nuevos)
update_data = {
    'id': cliente_id_update,
    'nombre': datos_antiguos[0],
    'email': datos_antiguos[1],
    'ciudad_anterior': datos_antiguos[2],
    'ciudad_nueva': nueva_ciudad
}
registrar_cambio('UPDATE', update_data)
conn_origen.commit()
time.sleep(1)

# --- Operación 3: DELETE (eliminar un cliente) ---
cliente_id_delete = 7  # Elena Díaz

# Obtener datos antes del delete para el log
cursor_origen.execute("SELECT nombre, email, ciudad FROM clientes WHERE id = ?", (cliente_id_delete,))
datos_eliminados = cursor_origen.fetchone()

# Realizar el delete
cursor_origen.execute("DELETE FROM clientes WHERE id = ?", (cliente_id_delete,))

# Registrar el cambio
delete_data = {
    'id': cliente_id_delete,
    'nombre': datos_eliminados[0],
    'email': datos_eliminados[1],
    'ciudad': datos_eliminados[2]
}
registrar_cambio('DELETE', delete_data)
conn_origen.commit()
time.sleep(1)

# --- Operación 4: Otro INSERT para tener más variedad ---
otro_cliente = {
    'nombre': 'Carmen Ruiz',
    'email': 'carmen.ruiz@email.com',
    'ciudad': 'Córdoba'
}
cursor_origen.execute(
    "INSERT INTO clientes (nombre, email, ciudad) VALUES (?, ?, ?)",
    (otro_cliente['nombre'], otro_cliente['email'], otro_cliente['ciudad'])
)
nuevo_id2 = cursor_origen.lastrowid
otro_cliente['id'] = nuevo_id2
registrar_cambio('INSERT', otro_cliente)
conn_origen.commit()

print("\n--- CAMBIOS SIMULADOS COMPLETADOS ---")

# Mostrar el estado final de la base de datos origen
df_clientes_origen_final = pd.read_sql_query("SELECT * FROM clientes", conn_origen)
print("\nEstado FINAL de la base de datos ORIGEN:")
display(df_clientes_origen_final)

# No cerramos la conexión aún

### **2.3. Visualizar el Archivo de Logs Generado**

Veamos cómo se ve nuestro archivo de logs, que será la entrada para nuestro proceso de CDC.

In [ ]:
# Leer y mostrar el contenido del archivo de logs
print("Contenido del archivo de logs 'cambios_clientes.log':")
print("-" * 80)
with open(log_file, 'r') as f:
    for linea in f:
        print(linea.strip())
print("-" * 80)

---
## **Parte 3: Procesar el Log y Aplicar Cambios a una Base de Datos Destino**

Ahora viene la parte clave del CDC: leer el archivo de logs (simulando la lectura del log transaccional) y aplicar esos cambios a una base de datos de destino (que podría ser un Data Lake, un Data Warehouse, o en nuestro caso, otra base de datos SQLite).

### **3.1. Crear la Base de Datos Destino Vacía**

Primero, creamos la base de datos destino (inicialmente vacía) con la misma estructura que la origen.

In [ ]:
# Conectar a la base de datos de destino (se creará si no existe)
conn_destino = sqlite3.connect('db_destino.db')
cursor_destino = conn_destino.cursor()

# Crear la tabla clientes en la base de datos destino (vacía inicialmente)
cursor_destino.execute('''
    CREATE TABLE IF NOT EXISTS clientes (
        id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        ciudad TEXT,
        fecha_registro TIMESTAMP,
        activo BOOLEAN DEFAULT 1
    )
''')

# Eliminar cualquier dato previo para empezar desde cero
cursor_destino.execute("DELETE FROM clientes")
conn_destino.commit()

# Verificar que la tabla destino está vacía
df_destino_vacio = pd.read_sql_query("SELECT * FROM clientes", conn_destino)
print("Base de datos DESTINO (inicialmente vacía):")
display(df_destino_vacio)
print(f"Total de registros: {len(df_destino_vacio)}")

### **3.2. Procesar el Log y Replicar los Cambios**

Ahora leeremos el archivo `cambios_clientes.log` línea por línea, y para cada operación registrada, aplicaremos el cambio correspondiente en la base de datos destino.

In [ ]:
def aplicar_cambio_en_destino(operacion, datos, cursor):
    """
    Aplica un cambio (INSERT, UPDATE, DELETE) a la base de datos destino.
    """
    try:
        if operacion == 'INSERT':
            # Para INSERT, insertamos el nuevo registro
            cursor.execute('''
                INSERT INTO clientes (id, nombre, email, ciudad)
                VALUES (?, ?, ?, ?)
            ''', (datos['id'], datos['nombre'], datos['email'], datos['ciudad']))
            print(f"  Aplicado INSERT: Cliente {datos['nombre']} (ID: {datos['id']})")

        elif operacion == 'UPDATE':
            # Para UPDATE, actualizamos el registro existente
            cursor.execute('''
                UPDATE clientes
                SET ciudad = ?
                WHERE id = ?
            ''', (datos['ciudad_nueva'], datos['id']))
            print(f"  Aplicado UPDATE: Cliente ID {datos['id']} - Ciudad cambió de '{datos['ciudad_anterior']}' a '{datos['ciudad_nueva']}'")

        elif operacion == 'DELETE':
            # Para DELETE, eliminamos el registro
            cursor.execute("DELETE FROM clientes WHERE id = ?", (datos['id'],))
            print(f"  Aplicado DELETE: Cliente {datos['nombre']} (ID: {datos['id']})")

    except Exception as e:
        print(f"  ERROR al aplicar {operacion}: {e}")
        # En un sistema real, esto se registraría y posiblemente se reintentaría

print("--- PROCESANDO LOG Y REPLICANDO CAMBIOS EN DESTINO ---")

# Leer el archivo de logs línea por línea
with open(log_file, 'r') as f:
    for linea in f:
        # Ignorar líneas de comentario (que empiezan con #)
        if linea.startswith('#') or linea.strip() == '':
            continue

        # Parsear la línea: timestamp|operacion|datos_json
        partes = linea.strip().split('|', 2)
        if len(partes) == 3:
            timestamp, operacion, datos_json = partes
            datos = json.loads(datos_json)

            print(f"\nProcesando operación: {operacion} (timestamp: {timestamp})")
            aplicar_cambio_en_destino(operacion, datos, cursor_destino)
        else:
            print(f"Línea con formato incorrecto ignorada: {linea}")

# Confirmar los cambios en la base de datos destino
conn_destino.commit()
print("\n--- REPLICACIÓN COMPLETADA ---")

# Mostrar el estado final de la base de datos destino
df_destino_final = pd.read_sql_query("SELECT * FROM clientes ORDER BY id", conn_destino)
print("\nEstado FINAL de la base de datos DESTINO (después de replicación):")
display(df_destino_final)

### **3.3. Verificación de la Sincronización**

Comparemos ahora la base de datos ORIGEN (que ha sufrido los cambios directamente) con la base de datos DESTINO (que ha sido actualizada a través de la lectura del log) para verificar que están sincronizadas.

In [ ]:
# Obtener datos actualizados de ambas bases
df_origen_verificacion = pd.read_sql_query("SELECT * FROM clientes ORDER BY id", conn_origen)
df_destino_verificacion = pd.read_sql_query("SELECT * FROM clientes ORDER BY id", conn_destino)

print("--- VERIFICACIÓN DE SINCRONIZACIÓN ---")

# Comparar DataFrames
if df_origen_verificacion.equals(df_destino_verificacion):
    print("✅ ÉXITO: Las bases de datos ORIGEN y DESTINO están sincronizadas.")
else:
    print("❌ ERROR: Las bases de datos NO están sincronizadas.")

# Mostrar diferencias si las hay
if not df_origen_verificacion.equals(df_destino_verificacion):
    print("\nDiferencias encontradas:")
    # Comparación simple: mostrar filas que no coinciden
    for idx in range(max(len(df_origen_verificacion), len(df_destino_verificacion))):
        if idx < len(df_origen_verificacion) and idx < len(df_destino_verificacion):
            if not df_origen_verificacion.iloc[idx].equals(df_destino_verificacion.iloc[idx]):
                print(f"  Fila {idx+1}:")
                print(f"    Origen:  {df_origen_verificacion.iloc[idx].to_dict()}")
                print(f"    Destino: {df_destino_verificacion.iloc[idx].to_dict()}")
        elif idx < len(df_origen_verificacion):
            print(f"  Fila {idx+1} solo en ORIGEN: {df_origen_verificacion.iloc[idx].to_dict()}")
        else:
            print(f"  Fila {idx+1} solo en DESTINO: {df_destino_verificacion.iloc[idx].to_dict()}")
else:
    print(f"\nAmbas bases contienen {len(df_origen_verificacion)} registros.")
    display(df_origen_verificacion)

# Cerrar conexiones
conn_origen.close()
conn_destino.close()
print("\nConexiones a bases de datos cerradas.")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora es tu turno de aplicar y extender lo aprendido.

### **Ejercicio 1: Añadir más operaciones al log**

Vuelve a la **Parte 2.2** (simulación de cambios) y añade al menos 3 operaciones más:
*   Un INSERT de un cliente nuevo.
*   Un UPDATE que modifique el nombre de un cliente (no solo la ciudad). Tendrás que modificar la estructura del log para incluir más campos.
*   Un DELETE de otro cliente.

Luego, ejecuta todas las celdas desde la Parte 2.2 en adelante para ver cómo tu pipeline de CDC replica estos nuevos cambios.

In [ ]:
# Aquí puedes experimentar con nuevas operaciones
# Vuelve a la celda de simulación de cambios y modifícala
# Opcional: puedes crear un nuevo bloque de código aquí que simule una segunda ronda de cambios

# --- INICIO DE TU CÓDIGO ---

# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Manejo de errores y reintentos**

Modifica la función `aplicar_cambio_en_destino` para que sea más robusta. Por ejemplo, si se intenta aplicar un UPDATE sobre un ID que no existe en destino, el programa actualmente fallará. Mejora la función para que:
1.  Verifique si el registro existe antes de aplicar un UPDATE o DELETE.
2.  Si no existe, en lugar de fallar, registre un mensaje de advertencia y quizás realice una acción alternativa (como insertarlo si es un UPDATE, o ignorarlo si es un DELETE).

*Pista: Puedes hacer una consulta SELECT previa para verificar la existencia del registro.*

In [ ]:
# Define aquí tu versión mejorada de la función y pruébala

def aplicar_cambio_en_destino_mejorado(operacion, datos, cursor):
    """
    Versión mejorada con manejo de errores.
    """
    # --- INICIO DE TU CÓDIGO ---
    pass
    # --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Simular un sistema de checkpointing**

En sistemas CDC reales, es importante saber hasta qué punto del log se ha procesado (para poder reanudar en caso de fallo). Simula un sistema simple de checkpointing:

1.  Después de procesar cada operación del log, escribe en un archivo `checkpoint.txt` la última línea del log que se ha procesado exitosamente (puedes guardar el timestamp o el número de línea).
2.  Modifica el procesamiento para que, si existe el archivo `checkpoint.txt`, comience a leer el log desde ese punto, ignorando las líneas ya procesadas.

In [ ]:
# Implementa aquí tu sistema de checkpointing

# --- INICIO DE TU CÓDIGO ---

# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 4: Reflexión sobre la integración con Kafka y Debezium**

Investiga un poco y responde en una celda Markdown:

*   ¿Cómo se integraría Debezium en el pipeline que hemos simulado? ¿Qué parte del proceso de CDC automatizaría?
*   ¿Cuál sería el rol de Apache Kafka en una arquitectura real de CDC?
*   ¿Qué ventajas tiene el CDC basado en logs (como el que simula nuestro archivo) sobre el basado en triggers?

**Tus respuestas:**

*   
*   
*   